# 03a — Projection Intuition

## The One-Sentence Version
Projecting data onto a line is casting a shadow — the best shadow preserves the most spread.

## What You'll Build Intuition For
- What "projection" means geometrically
- Why maximising variance = finding the best summary
- The trade-off between what's kept and what's lost
- How the first two principal components emerge from this logic

## Prerequisites
- [02a — Correlation and Redundancy](../02_redundancy_and_structure/02a_correlation_and_redundancy.ipynb)
- [02b — Intrinsic vs Ambient Dimension](../02_redundancy_and_structure/02b_intrinsic_vs_ambient.ipynb)

## The Story

Imagine shining a torch at a 3D object and looking at the shadow on the wall. The shadow is a 2D summary of a 3D thing. Some shadows are informative — you can tell it's a chair. Some are useless — just a blob.

Dimensionality reduction is the same idea. You have high-dimensional data and you want to "cast a shadow" onto fewer dimensions. The question is: which angle gives the most informative shadow?

The answer turns out to be beautifully simple: the best shadow is the one where the projected data is most spread out. Maximum spread, maximum information preserved. Minimum spread, and you've squashed your data into a useless line.

This notebook builds that intuition from scratch — starting with a 2D cloud and a line to project onto. By the end, you'll see exactly where principal components come from, before we ever touch an eigenvalue.

In [ ]:
import sys
import os
sys.path.insert(0, os.path.abspath('..'))

import numpy as np
import matplotlib.pyplot as plt
from matplotlib.patches import FancyArrowPatch

from utils.plotting import apply_style, COLOURS

apply_style()
rng = np.random.default_rng(42)

## Shadows on a Wall

Let's generate a 2D cloud of data that's elongated and tilted — like a cigar shape. Then we'll project it onto lines at different angles and see which angle captures the most spread.

In [ ]:
# Generate an elongated 2D cloud tilted at ~45 degrees
n = 200
angle_true = np.pi / 4  # 45 degrees

# Elongated in one direction, narrow in the other
spread_major = 3.0
spread_minor = 0.5

# Generate aligned data, then rotate
X_aligned = np.column_stack([
    rng.normal(0, spread_major, n),
    rng.normal(0, spread_minor, n),
])

# Rotation matrix
R = np.array([[np.cos(angle_true), -np.sin(angle_true)],
              [np.sin(angle_true),  np.cos(angle_true)]])
X = X_aligned @ R.T

fig, ax = plt.subplots(figsize=(8, 8))
ax.scatter(X[:, 0], X[:, 1], s=15, alpha=0.5, color=COLOURS["signal"])
ax.set_xlabel("Feature 1")
ax.set_ylabel("Feature 2")
ax.set_title("A tilted, elongated cloud of 2D data")
ax.set_aspect("equal")
ax.set_xlim(-8, 8)
ax.set_ylim(-8, 8)
plt.tight_layout()
plt.show()

print(f"Data shape: {X.shape}")
print(f"The cloud is stretched along ~45°. Which line captures the most spread?")

In [ ]:
# Project onto lines at different angles and show the resulting histograms
def project_onto_angle(X, theta):
    """Project 2D data onto a unit vector at angle theta."""
    v = np.array([np.cos(theta), np.sin(theta)])
    return X @ v, v

angles_deg = [0, 30, 45, 90, 135]
angles_rad = [np.deg2rad(a) for a in angles_deg]

fig, axes = plt.subplots(2, len(angles_deg), figsize=(16, 8),
                         gridspec_kw={"height_ratios": [3, 1]})

for i, (theta, deg) in enumerate(zip(angles_rad, angles_deg)):
    proj, v = project_onto_angle(X, theta)
    variance = np.var(proj)

    # Top row: scatter + projection line
    ax_scatter = axes[0, i]
    ax_scatter.scatter(X[:, 0], X[:, 1], s=8, alpha=0.3, color=COLOURS["signal"])

    # Draw projection line
    line_extent = 8
    ax_scatter.plot([-line_extent * v[0], line_extent * v[0]],
                    [-line_extent * v[1], line_extent * v[1]],
                    color=COLOURS["noise"], linewidth=2, alpha=0.8)

    # Draw projections as dots on the line
    proj_points = np.outer(proj, v)
    ax_scatter.scatter(proj_points[:, 0], proj_points[:, 1],
                       s=5, alpha=0.4, color=COLOURS["noise"])

    ax_scatter.set_xlim(-8, 8)
    ax_scatter.set_ylim(-8, 8)
    ax_scatter.set_aspect("equal")
    ax_scatter.set_title(f"{deg}° — var = {variance:.2f}")
    if i == 0:
        ax_scatter.set_ylabel("Feature 2")

    # Bottom row: histogram of projected values
    ax_hist = axes[1, i]
    ax_hist.hist(proj, bins=25, alpha=0.7, color=COLOURS["noise"],
                 edgecolor="white", linewidth=0.3)
    ax_hist.set_xlabel(f"Projection ({deg}°)")
    if i == 0:
        ax_hist.set_ylabel("Count")

fig.suptitle("Projecting onto different angles — which shadow has the most spread?",
             fontweight="bold")
plt.tight_layout()
plt.savefig("visuals/03a_shadow_angles.png", dpi=150, bbox_inches="tight")
plt.show()

The 45° projection has the highest variance — the histogram is the widest. That's because it aligns with the direction the data is most spread out. The 135° projection (perpendicular to it) has the smallest variance — all the points are squashed together.

> **Key insight:** The "best" projection line is the one that maximises the variance of the projected data. Maximum variance = maximum information preserved = the most informative shadow.

In [ ]:
# Variance as a function of angle — the full sweep
sweep_angles = np.linspace(0, np.pi, 200)
variances = []
for theta in sweep_angles:
    proj, _ = project_onto_angle(X, theta)
    variances.append(np.var(proj))

best_idx = np.argmax(variances)
best_angle_deg = np.rad2deg(sweep_angles[best_idx])

fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(np.rad2deg(sweep_angles), variances, color=COLOURS["signal"], linewidth=2)
ax.axvline(best_angle_deg, color=COLOURS["accent"], linestyle="--", linewidth=1.5,
           label=f"Max variance at {best_angle_deg:.0f}°")
ax.set_xlabel("Projection Angle (degrees)")
ax.set_ylabel("Variance of Projected Data")
ax.set_title("Variance vs Projection Angle")
ax.legend()
plt.tight_layout()
plt.savefig("visuals/03a_variance_sweep.png", dpi=150, bbox_inches="tight")
plt.show()

print(f"Maximum variance at {best_angle_deg:.1f}° — this is the first principal component direction.")
print(f"True data elongation was at {np.rad2deg(angle_true):.1f}°. PCA finds the shape.")

## From Shadow to Projection

Let's formalise what we just did.

A **projection** onto a unit vector $\mathbf{v}$ is just a dot product:

$$\text{projected value}_i = \mathbf{x}_i \cdot \mathbf{v}$$

For the whole dataset: $\mathbf{p} = X \mathbf{v}$ where $\|\mathbf{v}\| = 1$.

The **variance** of the projection is $\text{Var}(\mathbf{p}) = \mathbf{v}^T C \mathbf{v}$ where $C$ is the covariance matrix.

The direction that maximises $\mathbf{v}^T C \mathbf{v}$ subject to $\|\mathbf{v}\| = 1$ is the **eigenvector of $C$ with the largest eigenvalue**. That's PC1.

In [ ]:
# Verify: eigenvector of covariance = the angle we found by brute-force sweep
X_centered = X - X.mean(axis=0)
C = np.cov(X_centered.T)

eigenvalues, eigenvectors = np.linalg.eigh(C)
# Sort descending
idx = np.argsort(eigenvalues)[::-1]
eigenvalues = eigenvalues[idx]
eigenvectors = eigenvectors[:, idx]

pc1 = eigenvectors[:, 0]
pc2 = eigenvectors[:, 1]

print(f"Covariance matrix:\n{C.round(2)}\n")
print(f"Eigenvalues: {eigenvalues.round(3)}")
print(f"PC1 direction: [{pc1[0]:.3f}, {pc1[1]:.3f}]")
print(f"PC1 angle: {np.rad2deg(np.arctan2(pc1[1], pc1[0])):.1f}°")
print(f"Brute-force best angle: {best_angle_deg:.1f}°")
print(f"\nThey match — PCA is just finding the angle that maximises spread.")

## What Gets Lost

Projection throws away information. When you project a 2D point onto a line, you lose the distance from the point to the line. That distance is the **reconstruction error**.

Here's the beautiful trade-off: **maximising variance preserved = minimising reconstruction error.** They're two sides of the same coin.

In [ ]:
# Show projection + reconstruction error for a good and bad angle
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 6))

for ax, theta, label in [(ax1, np.arctan2(pc1[1], pc1[0]), "Best angle (PC1)"),
                          (ax2, np.arctan2(pc1[1], pc1[0]) + np.pi/2, "Worst angle (⊥ PC1)")]:
    v = np.array([np.cos(theta), np.sin(theta)])
    proj_scalar = X_centered @ v
    proj_points = np.outer(proj_scalar, v)
    residuals = X_centered - proj_points
    recon_error = np.mean(np.sum(residuals**2, axis=1))
    variance = np.var(proj_scalar)

    ax.scatter(X_centered[:, 0], X_centered[:, 1], s=10, alpha=0.3,
               color=COLOURS["signal"])

    # Projection line
    ax.plot([-8*v[0], 8*v[0]], [-8*v[1], 8*v[1]],
            color=COLOURS["accent"], linewidth=2, alpha=0.8)

    # Draw residual lines for a subset
    for k in range(0, n, 8):
        ax.plot([X_centered[k, 0], proj_points[k, 0]],
                [X_centered[k, 1], proj_points[k, 1]],
                color=COLOURS["noise"], alpha=0.3, linewidth=0.5)

    ax.scatter(proj_points[:, 0], proj_points[:, 1], s=5, alpha=0.5,
               color=COLOURS["accent"])

    ax.set_xlim(-8, 8)
    ax.set_ylim(-8, 8)
    ax.set_aspect("equal")
    ax.set_title(f"{label}\nVariance: {variance:.2f}  |  Recon error: {recon_error:.2f}")

fig.suptitle("Max variance = Min reconstruction error", fontweight="bold")
plt.tight_layout()
plt.savefig("visuals/03a_reconstruction_error.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
# Prove the trade-off: variance + recon_error = total variance (constant)
total_var = np.trace(C)
sweep_recon = []
sweep_var = []

for theta in sweep_angles:
    v = np.array([np.cos(theta), np.sin(theta)])
    proj_scalar = X_centered @ v
    proj_points = np.outer(proj_scalar, v)
    residuals = X_centered - proj_points

    sweep_var.append(np.var(proj_scalar))
    sweep_recon.append(np.mean(np.sum(residuals**2, axis=1)))

fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(np.rad2deg(sweep_angles), sweep_var, color=COLOURS["signal"],
        linewidth=2, label="Variance preserved")
ax.plot(np.rad2deg(sweep_angles), sweep_recon, color=COLOURS["noise"],
        linewidth=2, label="Reconstruction error")
ax.axhline(total_var, color=COLOURS["neutral"], linestyle=":", alpha=0.5,
           label=f"Total variance = {total_var:.2f}")
ax.set_xlabel("Projection Angle (degrees)")
ax.set_ylabel("Value")
ax.set_title("The Trade-Off: Variance Preserved + Error = Constant")
ax.legend()
plt.tight_layout()
plt.savefig("visuals/03a_tradeoff.png", dpi=150, bbox_inches="tight")
plt.show()

print(f"At every angle: variance + recon_error ≈ {total_var:.2f} (total variance)")
print("Maximising one automatically minimises the other.")

> **Key insight:** You don't have a choice between "keep the variance" and "reduce the error" — they're the same optimisation. Finding the maximum-variance direction *is* finding the minimum-error direction.

## Multiple Lines

We found the best single line (PC1). What's the best *second* line?

It must be perpendicular (orthogonal) to PC1. Why? Because PC1 already captured all the variance in its direction. Any non-perpendicular line would partially re-capture the same variance. The perpendicular direction captures whatever's left.

In 2D there's only one choice for PC2 — the perpendicular. In higher dimensions, you'd pick the direction of maximum remaining variance among all directions orthogonal to PC1.

In [ ]:
# Show the two principal components on the data
fig, (ax1, ax2, ax3) = plt.subplots(1, 3, figsize=(16, 5))

# Left: original data with PC arrows
ax1.scatter(X_centered[:, 0], X_centered[:, 1], s=10, alpha=0.4,
            color=COLOURS["signal"])

scale1 = np.sqrt(eigenvalues[0]) * 2
scale2 = np.sqrt(eigenvalues[1]) * 2
ax1.annotate("", xy=pc1 * scale1, xytext=(0, 0),
             arrowprops=dict(arrowstyle="->", color=COLOURS["noise"], lw=2.5))
ax1.annotate("", xy=pc2 * scale2, xytext=(0, 0),
             arrowprops=dict(arrowstyle="->", color=COLOURS["accent"], lw=2.5))
ax1.text(pc1[0]*scale1*1.15, pc1[1]*scale1*1.15, "PC1",
         fontsize=12, fontweight="bold", color=COLOURS["noise"])
ax1.text(pc2[0]*scale2*1.3, pc2[1]*scale2*1.3, "PC2",
         fontsize=12, fontweight="bold", color=COLOURS["accent"])
ax1.set_xlim(-8, 8)
ax1.set_ylim(-8, 8)
ax1.set_aspect("equal")
ax1.set_title("Original data + principal components")

# Middle: project onto PC1 only
proj_pc1 = X_centered @ pc1
ax2.hist(proj_pc1, bins=30, alpha=0.7, color=COLOURS["noise"],
         edgecolor="white", linewidth=0.3)
ax2.set_xlabel("PC1 score")
ax2.set_title(f"PC1: var = {eigenvalues[0]:.2f} ({eigenvalues[0]/eigenvalues.sum():.0%})")

# Right: project onto PC2 only
proj_pc2 = X_centered @ pc2
ax3.hist(proj_pc2, bins=30, alpha=0.7, color=COLOURS["accent"],
         edgecolor="white", linewidth=0.3)
ax3.set_xlabel("PC2 score")
ax3.set_title(f"PC2: var = {eigenvalues[1]:.2f} ({eigenvalues[1]/eigenvalues.sum():.0%})")

fig.suptitle("PC1 captures the spread, PC2 captures what's left", fontweight="bold")
plt.tight_layout()
plt.savefig("visuals/03a_two_components.png", dpi=150, bbox_inches="tight")
plt.show()

print(f"PC1 captures {eigenvalues[0]/eigenvalues.sum():.1%} of total variance")
print(f"PC2 captures {eigenvalues[1]/eigenvalues.sum():.1%} of total variance")
print(f"Together: {eigenvalues.sum()/eigenvalues.sum():.1%} (all of it — we're in 2D)")

In [ ]:
# Project to PC space and show the rotated data
W = eigenvectors  # columns are PC1, PC2
X_pca = X_centered @ W

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))

ax1.scatter(X_centered[:, 0], X_centered[:, 1], s=10, alpha=0.5,
            color=COLOURS["signal"])
ax1.set_xlabel("Feature 1")
ax1.set_ylabel("Feature 2")
ax1.set_title("Original (correlated)")
ax1.set_aspect("equal")
ax1.set_xlim(-8, 8)
ax1.set_ylim(-8, 8)

ax2.scatter(X_pca[:, 0], X_pca[:, 1], s=10, alpha=0.5,
            color=COLOURS["signal"])
ax2.set_xlabel("PC1")
ax2.set_ylabel("PC2")
ax2.set_title("PCA-rotated (uncorrelated)")
ax2.set_aspect("equal")
ax2.set_xlim(-8, 8)
ax2.set_ylim(-8, 8)

fig.suptitle("PCA is a rotation that aligns the data with its axes of variation",
             fontweight="bold")
plt.tight_layout()
plt.savefig("visuals/03a_rotation.png", dpi=150, bbox_inches="tight")
plt.show()

print("Left: features are correlated (tilted cloud).")
print("Right: PC axes are uncorrelated (axis-aligned cloud).")
print("PCA didn't change the data — it just rotated the axes to align with the spread.")

<details>
<summary><b>The Maths Behind This</b></summary>

**Projection:** Given data matrix $X \in \mathbb{R}^{n \times d}$ (centred) and unit vector $\mathbf{v}$, the projected values are $\mathbf{p} = X\mathbf{v}$ and the variance of the projection is:

$$\text{Var}(\mathbf{p}) = \mathbf{v}^T C \mathbf{v}$$

where $C = \frac{1}{n-1} X^T X$ is the covariance matrix.

**Optimisation:** We want to maximise $\mathbf{v}^T C \mathbf{v}$ subject to $\|\mathbf{v}\| = 1$. Using Lagrange multipliers:

$$\nabla_\mathbf{v} (\mathbf{v}^T C \mathbf{v} - \lambda(\mathbf{v}^T\mathbf{v} - 1)) = 0$$
$$\Rightarrow C\mathbf{v} = \lambda \mathbf{v}$$

This is the eigenvalue equation. The maximum variance $\lambda$ is achieved by the eigenvector with the largest eigenvalue.

**Reconstruction error:** The mean squared distance from each point to its projection is:

$$E = \frac{1}{n} \sum_i \|\mathbf{x}_i - (\mathbf{x}_i \cdot \mathbf{v})\mathbf{v}\|^2 = \text{tr}(C) - \mathbf{v}^T C \mathbf{v}$$

Since $\text{tr}(C)$ is constant, minimising $E$ is equivalent to maximising $\mathbf{v}^T C \mathbf{v}$.

</details>

## Where This Matters

**Clinical data dashboards:** A hospital tracks 50 patient vitals. Projecting onto the first 2-3 principal components gives a compact summary that can be plotted on a screen. Outliers in this projected space correspond to patients whose vital sign *pattern* is unusual — more informative than any single vital crossing a threshold.

**Sensor fusion:** Multiple sensors measuring related quantities (temperature, pressure, flow rate) produce correlated signals. Projecting onto the first principal component gives a single "system health" score that summarises the redundant measurements, reducing noise in the process.

## Feynman Check

1. **Explain PCA to someone using only the word "shadow."**

2. **Why does the first principal component have the largest variance?**

3. **If you project 3D data onto a plane, what does the reconstruction error represent physically?**

You now understand the geometric intuition behind PCA — finding the best shadows. In **03b — PCA From Scratch**, we'll build the full algorithm with numpy, step by step.